# 04 – Feature Engineering: Variables Educativas

**Proyecto:** Predicción de Subempleo por Insuficiencia de Horas — EPEN 2024  
**Etapa:** Feature Engineering – Paso 2 de 4  
**Dataset de entrada:** `data/feature_engineering/epen_fe_demographic.csv`  
**Dataset de salida:** `data/feature_engineering/epen_fe_education.csv`

## Objetivo

Crear variables derivadas del **nivel educativo** del trabajador usando las variables de la EPEN 2024: `C366` (último nivel educativo alcanzado), `C366_1` (nivel en curso o interrumpido) y `C366_2` (año o grado cursado).

La educación es un predictor clave del subempleo: trabajadores con menor nivel educativo tienden a ocupar empleos de menos horas o a aceptar trabajos por debajo de su capacidad.

### Codificación de C366 (nivel educativo alcanzado)

| Código | Nivel |
|:------:|:------|
| 1 | Sin nivel |
| 2 | Inicial |
| 3 | Primaria incompleta |
| 4 | Primaria completa |
| 5 | Secundaria incompleta |
| 6 | Secundaria completa |
| 7 | Superior no universitaria incompleta |
| 8 | Superior no universitaria completa |
| 9 | Superior universitaria incompleta |
| 10 | Superior universitaria completa |
| 11 | Post-grado |
| 12 | Maestría / Doctorado |

### Restricciones
- No se usa: `P209H`, `C333`, `C334`, `fa_son24`
- No se realiza train/test split en este notebook

---
## 1. Cargar Librerías

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 80)
pd.set_option('display.float_format', '{:.4f}'.format)

print('Librerías cargadas correctamente.')

Librerías cargadas correctamente.


---
## 2. Cargar Dataset

In [2]:
INPUT_PATH = Path('../data/feature_engineering/epen_fe_demographic.csv')

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f'No se encontró el archivo de entrada: {INPUT_PATH}\n'
        'Ejecuta primero: 04_feature_engineering/01_demographic_features.ipynb'
    )

df = pd.read_csv(INPUT_PATH, low_memory=False)
print(f'Dataset cargado : {INPUT_PATH.name}')
print(f'Dimensiones     : {df.shape[0]:,} filas x {df.shape[1]} columnas')

Dataset cargado : epen_fe_demographic.csv
Dimensiones     : 24,054 filas x 61 columnas


---
## 3. Validación Inicial

In [3]:
assert 'target_subempleo_horas' in df.columns, \
    "ERROR: 'target_subempleo_horas' no encontrado."
assert df['target_subempleo_horas'].isnull().sum() == 0, \
    'ERROR: target_subempleo_horas contiene nulos.'
print('target_subempleo_horas presente y sin nulos: OK')

for lv in ['P209H', 'C333', 'C334']:
    assert lv not in df.columns, f'ERROR: variable de leakage {lv} encontrada.'
print('Variables de leakage ausentes: OK')

print('\nDistribución del target:')
counts = df['target_subempleo_horas'].value_counts()
pct    = df['target_subempleo_horas'].value_counts(normalize=True) * 100
display(pd.DataFrame({'conteo': counts, 'porcentaje (%)': pct.round(2)}))

target_subempleo_horas presente y sin nulos: OK
Variables de leakage ausentes: OK

Distribución del target:


,conteo,porcentaje (%)
target_subempleo_horas,,
0,18064,75.1000
1,5990,24.9000


---
## 4. Crear Copia de Trabajo

In [4]:
df_fe = df.copy()
n_original = df_fe.shape[0]
features_created = []

print(f'Copia creada: {df_fe.shape[0]:,} filas x {df_fe.shape[1]} columnas')

Copia creada: 24,054 filas x 61 columnas


---
## 5. Nivel Educativo Ordinal

Convierte `C366` en una variable ordinal numérica (1–12) para modelos que puedan aprovechar el orden.

**Feature creado:** `nivel_educativo_ord`

In [5]:
if 'C366' in df_fe.columns:
    df_fe['nivel_educativo_ord'] = pd.to_numeric(df_fe['C366'], errors='coerce')
    features_created.append('nivel_educativo_ord')

    print('Distribución de nivel_educativo_ord (C366):')
    print(df_fe['nivel_educativo_ord'].value_counts().sort_index())
    print(f"\nNulos: {df_fe['nivel_educativo_ord'].isnull().sum()}")
else:
    print('ADVERTENCIA: C366 no encontrado. Variable nivel_educativo_ord no creada.')

Distribución de nivel_educativo_ord (C366):
nivel_educativo_ord
1      134
2       19
3      688
4      925
5     2340
6     8030
7       23
8     1872
9     3305
10    2051
11    3899
12     768
Name: count, dtype: int64

Nulos: 0


---
## 6. Variables Educativas Binarias

**Features creados:**
- `educacion_basica_o_menos`: nivel ≤ 5 (sin nivel hasta secundaria incompleta)
- `secundaria_completa`: nivel == 6
- `superior_incompleta`: nivel en {7, 9} (no universitaria o universitaria, incompleta)
- `superior_completa`: nivel en {8, 10, 11, 12} (no universitaria completa, universitaria completa, posgrado)
- `universitaria_completa_o_mas`: nivel en {10, 11, 12}
- `educacion_superior`: nivel ≥ 7 (cualquier educación superior)
- `brecha_educativa_baja`: nivel ≤ 4 (sin nivel hasta primaria completa)

In [6]:
if 'nivel_educativo_ord' in df_fe.columns:
    edu = df_fe['nivel_educativo_ord']

    df_fe['educacion_basica_o_menos']    = (edu <= 5).astype(int)
    df_fe['secundaria_completa']         = (edu == 6).astype(int)
    df_fe['superior_incompleta']         = edu.isin([7, 9]).astype(int)
    df_fe['superior_completa']           = edu.isin([8, 10, 11, 12]).astype(int)
    df_fe['universitaria_completa_o_mas']= edu.isin([10, 11, 12]).astype(int)
    df_fe['educacion_superior']          = (edu >= 7).astype(int)
    df_fe['brecha_educativa_baja']       = (edu <= 4).astype(int)

    nuevas = [
        'educacion_basica_o_menos', 'secundaria_completa', 'superior_incompleta',
        'superior_completa', 'universitaria_completa_o_mas', 'educacion_superior',
        'brecha_educativa_baja'
    ]
    features_created += nuevas

    print('Distribución de variables educativas binarias:')
    for col in nuevas:
        pct = df_fe[col].mean() * 100
        print(f'  {col:<35}: {df_fe[col].sum():>7,}  ({pct:.1f}%)')
else:
    print('ADVERTENCIA: nivel_educativo_ord no disponible. Variables educativas binarias no creadas.')

Distribución de variables educativas binarias:
  educacion_basica_o_menos           :   4,106  (17.1%)
  secundaria_completa                :   8,030  (33.4%)
  superior_incompleta                :   3,328  (13.8%)
  superior_completa                  :   8,590  (35.7%)
  universitaria_completa_o_mas       :   6,718  (27.9%)
  educacion_superior                 :  11,918  (49.5%)
  brecha_educativa_baja              :   1,766  (7.3%)


---
## 7. Grupo Educativo (Categórico)

Variable categórica textual para interpretación y análisis exploratorio.

**Feature creado:** `grupo_educativo`

In [7]:
if 'nivel_educativo_ord' in df_fe.columns:
    def asignar_grupo_educativo(nivel):
        if pd.isnull(nivel):
            return 'Sin datos'
        nivel = int(nivel)
        if nivel <= 2:
            return 'Sin nivel / Inicial'
        elif nivel <= 4:
            return 'Primaria'
        elif nivel <= 5:
            return 'Secundaria incompleta'
        elif nivel == 6:
            return 'Secundaria completa'
        elif nivel <= 8:
            return 'Superior no universitaria'
        elif nivel <= 10:
            return 'Superior universitaria'
        else:
            return 'Posgrado'

    df_fe['grupo_educativo'] = df_fe['nivel_educativo_ord'].apply(asignar_grupo_educativo)
    features_created.append('grupo_educativo')

    print('Distribución de grupo_educativo:')
    print(df_fe['grupo_educativo'].value_counts())
else:
    print('ADVERTENCIA: nivel_educativo_ord no disponible.')

Distribución de grupo_educativo:
grupo_educativo
Secundaria completa          8030
Superior universitaria       5356
Posgrado                     4667
Secundaria incompleta        2340
Superior no universitaria    1895
Primaria                     1613
Sin nivel / Inicial           153
Name: count, dtype: int64


---
## 8. Validación Cruzada con Target

In [8]:
if 'grupo_educativo' in df_fe.columns:
    print('Tasa de subempleo por grupo educativo:')
    tabla = df_fe.groupby('grupo_educativo')['target_subempleo_horas'].agg(
        conteo='count',
        tasa_subempleo='mean'
    ).sort_values('tasa_subempleo', ascending=False)
    tabla['tasa_subempleo'] = tabla['tasa_subempleo'].round(4)
    display(tabla)

if 'nivel_educativo_ord' in df_fe.columns:
    print('\nCorrelacion nivel_educativo_ord vs target:')
    corr = df_fe['nivel_educativo_ord'].corr(df_fe['target_subempleo_horas'])
    print(f'  Pearson r = {corr:.4f}')

Tasa de subempleo por grupo educativo:


,conteo,tasa_subempleo
grupo_educativo,,
Secundaria incompleta,2340,0.2910
Secundaria completa,8030,0.2659
Primaria,1613,0.2604
Superior no universitaria,1895,0.2480
Superior universitaria,5356,0.2300
Posgrado,4667,0.2203
Sin nivel / Inicial,153,0.1569



Correlacion nivel_educativo_ord vs target:
  Pearson r = -0.0455


---
## 9. Validaciones Finales

In [9]:
assert df_fe.shape[0] == n_original, \
    f'ERROR: el número de filas cambió. Original: {n_original}, actual: {df_fe.shape[0]}'
print(f'Número de filas sin cambios : OK ({n_original:,})')

assert 'target_subempleo_horas' in df_fe.columns
print('target_subempleo_horas presente: OK')

for lv in ['P209H', 'C333', 'C334']:
    assert lv not in df_fe.columns, f'ERROR: {lv} presente.'
print('Variables de leakage ausentes: OK')

print(f'\nFeatures educativos creados ({len(features_created)}):')
for feat in features_created:
    nulos = df_fe[feat].isnull().sum()
    print(f'  {feat:<38}: {nulos:>6} nulos  ({nulos/len(df_fe)*100:.2f}%)')

print(f'\nDimensiones finales  : {df_fe.shape[0]:,} filas x {df_fe.shape[1]} columnas')
print(f'Columnas nuevas      : {df_fe.shape[1] - df.shape[1]}')

Número de filas sin cambios : OK (24,054)
target_subempleo_horas presente: OK
Variables de leakage ausentes: OK

Features educativos creados (9):
  nivel_educativo_ord                   :      0 nulos  (0.00%)
  educacion_basica_o_menos              :      0 nulos  (0.00%)
  secundaria_completa                   :      0 nulos  (0.00%)
  superior_incompleta                   :      0 nulos  (0.00%)
  superior_completa                     :      0 nulos  (0.00%)
  universitaria_completa_o_mas          :      0 nulos  (0.00%)
  educacion_superior                    :      0 nulos  (0.00%)
  brecha_educativa_baja                 :      0 nulos  (0.00%)
  grupo_educativo                       :      0 nulos  (0.00%)

Dimensiones finales  : 24,054 filas x 70 columnas
Columnas nuevas      : 9


---
## 10. Guardar Resultados

In [10]:
OUTPUT_DIR = Path('../data/feature_engineering')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Dataset con features educativas
out_main = OUTPUT_DIR / 'epen_fe_education.csv'
df_fe.to_csv(out_main, index=False)
print(f'Dataset guardado : {out_main}  ({df_fe.shape[0]:,} x {df_fe.shape[1]})')

# Reporte de features creados
report = pd.DataFrame({
    'feature'   : features_created,
    'tipo_dato' : [str(df_fe[f].dtype) for f in features_created],
    'nulos'     : [df_fe[f].isnull().sum() for f in features_created],
    'n_unicos'  : [df_fe[f].nunique() for f in features_created],
    'pct_nulos' : [round(df_fe[f].isnull().mean() * 100, 4) for f in features_created],
})
out_report = OUTPUT_DIR / 'education_features_created.csv'
report.to_csv(out_report, index=False)
print(f'Reporte guardado : {out_report}  ({len(report)} features)')

print('\nTodos los archivos guardados correctamente.')
print('Siguiente paso -> 03_employment_features.ipynb')

Dataset guardado : ..\data\feature_engineering\epen_fe_education.csv  (24,054 x 70)
Reporte guardado : ..\data\feature_engineering\education_features_created.csv  (9 features)

Todos los archivos guardados correctamente.
Siguiente paso -> 03_employment_features.ipynb
